# Data Processing Pipeline

Prepares the Crunchbase 2013 snapshot (Kaggle: `justinas/startup-investments`) for use in the TextGrad-optimized VC investment agent.

**Pipeline steps:**
1. Load merged dataset from parquet
2. Verify data integrity (duplicates)
3. Filter to companies with known outcomes (`acquired`, `ipo`, `closed`)
4. Filter to companies founded between 2005 and 2013 (inclusive)
5. Keep all geographies (no country filter in this processing phase)
6. Filter to companies with substantive overview text (>= 300 characters)
7. Create binary target variable
8. Detect leakage risk in narrative fields (`overview`, `short_description`, `description`)
9. Create anonymized text features (`overview_anon`, `short_description_anon`)
12. Inspect missingness and apply non-statistical column pruning
13. Save cleaned dataset and run reproducibility checks

## Processing Assumptions (2013 Snapshot)

- This is a fixed historical snapshot; labels are defined only from status observed by 2013.
- `operating` is excluded because outcome is unresolved at snapshot time.
- All geographies are retained intentionally in this phase.
- No train/test-time statistical transforms are applied here; those are deferred to the training pipeline to avoid leakage.
- Leakage-prone columns (for example outcome-timing fields) are removed before final save.

## Imports and Reproducibility Setup

In [55]:
import pandas as pd 
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Load and Merge Dataset

In [56]:
# Load the data
companies = pd.read_csv("../data/raw/companies.csv", low_memory=False)
financial_orgs = pd.read_csv("../data/raw/financial_orgs.csv", low_memory=False)
rounds = pd.read_csv("../data/raw/funding_rounds.csv", low_memory=False)
investments = pd.read_csv("../data/raw/investments.csv", low_memory=False)

In [57]:
# Join the dataframes on the company id
companies = companies.rename(columns={'id': 'object_id'})

# --- Aggregate rounds → one row per company ---
rounds_agg = rounds.groupby('object_id').agg(
    num_rounds        = ('id', 'count'),
    total_raised_usd  = ('raised_amount_usd', 'sum'),
    first_funding_at  = ('funded_at', 'min'),
    last_funding_at   = ('funded_at', 'max'),
    round_types       = ('funding_round_type', lambda x: list(x.dropna().unique()))
).reset_index()

# --- Aggregate investments → one row per company ---
investments_agg = investments.groupby('funded_object_id').agg(
    num_investors     = ('investor_object_id', 'nunique'),
    num_invest_rounds = ('funding_round_id', 'nunique')
).reset_index().rename(columns={'funded_object_id': 'object_id'})

# --- Now merge cleanly (all 1:1 at this point) ---
df = companies.merge(rounds_agg,      on='object_id', how='left')
df = df.merge(investments_agg,      on='object_id', how='left')

# --- Verify: should equal len(companies) ---
assert len(df) == len(companies), f"Duplicates detected! {len(df)} vs {len(companies)}"

In [58]:
# Inspect resulting dataframe
print(companies.shape)
print(companies.columns.tolist())
print(companies.dtypes)

# Convert to parquet
companies.to_parquet("../data/processed/combined.parquet", index=False, engine="fastparquet")

# Read back from parquet (much faster for subsequent loads)
df = pd.read_parquet("../data/processed/combined.parquet")

(196553, 40)
['object_id', 'entity_type', 'entity_id', 'parent_id', 'name', 'normalized_name', 'permalink', 'category_code', 'status', 'founded_at', 'closed_at', 'domain', 'homepage_url', 'twitter_username', 'logo_url', 'logo_width', 'logo_height', 'short_description', 'description', 'overview', 'tag_list', 'country_code', 'state_code', 'city', 'region', 'first_investment_at', 'last_investment_at', 'investment_rounds', 'invested_companies', 'first_funding_at', 'last_funding_at', 'funding_rounds', 'funding_total_usd', 'first_milestone_at', 'last_milestone_at', 'milestones', 'relationships', 'created_by', 'created_at', 'updated_at']
object_id                  str
entity_type                str
entity_id                int64
parent_id              float64
name                       str
normalized_name            str
permalink                  str
category_code              str
status                     str
founded_at                 str
closed_at                  str
domain              

## Team Features

Two-stage aggregation pipeline to add team/education signal from `relationships.csv` and `degrees.csv` without causing row explosion.

**Stages:**
1. Build `company_person_edges` - deduplicated `(company_id, person_id)` edges.
2. Build `person_degree_features` - one row per person with degree counts and top-university indicator.
3. Aggregate to `company_team_features` - one row per company.
4. Left-join into `df` (see bottom of notebook, before save).

Temporal filter: only relationships/degrees plausibly known by 2013 are included.

In [59]:
import re
from pathlib import Path

SNAPSHOT_YEAR = 2013
TOP_UNIVERSITY_COUNT = 50


def _resolve_data_path(*candidates: str) -> str:
    for candidate in candidates:
        candidate_path = Path(candidate)
        if candidate_path.exists():
            return str(candidate_path)
    raise FileNotFoundError(f"None of these paths exist: {candidates}")


QS_RANKINGS_PATH = _resolve_data_path(
    "../data/raw/2024 QS World University Rankings.csv",
    "data/raw/2024 QS World University Rankings.csv",
)


def load_top_universities(rankings_path: str, top_n: int = 50) -> set[str]:
    rankings = pd.read_csv(rankings_path, low_memory=False)

    rank_cols = [col for col in rankings.columns if "RANK" in col.upper()]
    rank_col = next((col for col in rank_cols if "2024" in col.upper()), rank_cols[0] if rank_cols else None)
    if rank_col is None:
        raise ValueError(f"No rank column found in {rankings_path}")

    institution_col = next((col for col in rankings.columns if "INSTITUTION" in col.upper()), None)
    if institution_col is None:
        raise ValueError(f"No institution column found in {rankings_path}")

    top_rankings = rankings.copy()
    top_rankings[rank_col] = pd.to_numeric(
        top_rankings[rank_col].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce",
    )
    top_rankings = top_rankings.dropna(subset=[rank_col, institution_col])
    top_rankings = top_rankings.sort_values(rank_col)
    top_rankings = top_rankings[top_rankings[rank_col] <= top_n]

    return {
        str(name).strip().lower()
        for name in top_rankings[institution_col].dropna().tolist()
        if str(name).strip()
    }


TOP_UNIVERSITIES = load_top_universities(QS_RANKINGS_PATH, top_n=TOP_UNIVERSITY_COUNT)
print(f"Loaded {len(TOP_UNIVERSITIES)} top universities from {QS_RANKINGS_PATH}")

# ── Load raw tables ────────────────────────────────────────────────────────────
rels_raw = pd.read_csv(_resolve_data_path("../data/raw/relationships.csv", "data/raw/relationships.csv"), low_memory=False)
degrees_raw = pd.read_csv(_resolve_data_path("../data/raw/degrees.csv", "data/raw/degrees.csv"), low_memory=False)

# ── Stage 1: company-person edges ─────────────────────────────────────────────
# Temporal filter: drop relationships that started after snapshot year
rels_raw['start_at'] = pd.to_datetime(rels_raw['start_at'], errors='coerce')
rels_co = rels_raw[
    rels_raw['relationship_object_id'].str.startswith('c:', na=False) &
    (rels_raw['start_at'].dt.year.fillna(0).le(SNAPSHOT_YEAR) | rels_raw['start_at'].isna())
].copy()
rels_co = rels_co.rename(columns={
    'relationship_object_id': 'company_id',
    'person_object_id': 'person_id',
})
# Deduplicate: one row per (company, person)
company_person_edges = rels_co[['company_id', 'person_id']].drop_duplicates().copy()

print(f"company_person_edges: {len(company_person_edges):,} rows | "
      f"{company_person_edges['company_id'].nunique():,} unique companies | "
      f"{company_person_edges['person_id'].nunique():,} unique people")

# ── Stage 2: person-level degree features ─────────────────────────────────────
# Temporal filter: exclude degrees graduated after snapshot year
degrees_raw['graduated_at'] = pd.to_datetime(degrees_raw['graduated_at'], errors='coerce')
degrees_filt = degrees_raw[
    degrees_raw['graduated_at'].dt.year.fillna(0).le(SNAPSHOT_YEAR) |
    degrees_raw['graduated_at'].isna()
].copy()

degrees_filt['institution_lower'] = degrees_filt['institution'].fillna('').str.lower().str.strip()
degrees_filt['is_top_university'] = degrees_filt['institution_lower'].apply(
    lambda inst: int(any(u in inst for u in TOP_UNIVERSITIES)) if inst else 0
)

person_degree_features = (
    degrees_filt.groupby('object_id', as_index=False).agg(
        degree_count=('id', 'count'),
        top_university_degree_count=('is_top_university', 'sum'),
    )
    .rename(columns={'object_id': 'person_id'})
)
person_degree_features['has_any_degree'] = (person_degree_features['degree_count'] > 0).astype(int)
person_degree_features['has_top_university'] = (person_degree_features['top_university_degree_count'] > 0).astype(int)

print(f"person_degree_features: {len(person_degree_features):,} people with degree records")

# ── Stage 3: join person features onto edges, aggregate to company ────────────
edges_enriched = company_person_edges.merge(person_degree_features, on='person_id', how='left')
for col in ['degree_count', 'top_university_degree_count', 'has_any_degree', 'has_top_university']:
    edges_enriched[col] = edges_enriched[col].fillna(0).astype(int)

company_team_features = edges_enriched.groupby('company_id', as_index=False).agg(
    team_size=('person_id', 'nunique'),
    person_with_degree_count=('has_any_degree', 'sum'),
    any_top_university_person=('has_top_university', 'max'),
    top_university_person_count=('has_top_university', 'sum'),
)
company_team_features['person_with_degree_pct'] = (
    company_team_features['person_with_degree_count']
    / company_team_features['team_size'].replace(0, float('nan'))
).fillna(0.0)

assert company_team_features['company_id'].is_unique, "Duplicate company_id in company_team_features!"
print(f"company_team_features: {len(company_team_features):,} companies (one row per company)")
print(company_team_features[['team_size', 'person_with_degree_count',
                              'any_top_university_person', 'person_with_degree_pct']].describe().round(2))

Loaded 50 top universities from ../data/raw/2024 QS World University Rankings.csv
company_person_edges: 360,228 rows | 130,378 unique companies | 187,931 unique people
person_degree_features: 68,450 people with degree records
company_team_features: 130,378 companies (one row per company)
       team_size  person_with_degree_count  any_top_university_person  \
count  130378.00                 130378.00                  130378.00   
mean        2.76                      1.52                       0.21   
std         8.51                      7.20                       0.40   
min         1.00                      0.00                       0.00   
25%         1.00                      0.00                       0.00   
50%         1.00                      1.00                       0.00   
75%         3.00                      1.00                       0.00   
max      1093.00                    930.00                       1.00   

       person_with_degree_pct  
count               1

### Examine Duplicates

In [60]:
# Find rows where id is duplicated
duplicate_ids = df[df.duplicated(subset=["object_id"], keep=False)]["object_id"].unique()
duplicate_names = companies[companies.duplicated(subset=['name'], keep=False)]['name'].nunique()
print(f"Number of duplicate ids: {len(duplicate_ids)}")
print(f"Example duplicate ids: {duplicate_ids[:5]}")
print(f"Number of duplicate names: {duplicate_names}")

# Find differences between rows with the same id
for dup_id in duplicate_ids[:5]:  # Check first 5 duplicate ids
    dup_rows = df[df["id"] == dup_id]
    print(f"\nDuplicate rows for id {dup_id}:")
    print(dup_rows)

Number of duplicate ids: 0
Example duplicate ids: []
Number of duplicate names: 46


In [61]:
# Remove duplicate name rows
df = df.drop_duplicates(subset=['name'], keep='first').copy() 

In [62]:
len(df)

196506

## Filter rows based on `status` and founding year
**Success** = `acquired` or `ipo`
**Failure** = `closed`
**Exclude** = `operating` (outcome unknown at snapshot time)

Pre-2000 companies have had more than a decade to exit by the 2013 snapshot date, so the cohort is dominated by time artifacts: almost every company that was going to succeed already had, while the rest are mostly still marked `operating` and disappear before training. A 99% success rate for 1990–94 companies is not a VC signal, it is a snapshot artifact.

Filtering to 2005–2013 is the more realistic boundary. It brings the apparent success rate down to roughly 63%, keeps companies in a 0–8 year decision horizon at snapshot time, and gives the agent a genuinely hard classification problem instead of an easy old-company-exit heuristic.

In [63]:
# Filter rows with KNOWN outcomes only (status = acquired / ipo / closed)
df = df[df['status'].isin(['acquired', 'ipo', 'closed'])]
len(df)

13112

In [64]:
# Number of successful vs unsuccessful outcomes
df['status'].value_counts()

status
acquired    9394
closed      2584
ipo         1134
Name: count, dtype: int64

In [65]:
# Restrict to cohorts that were plausibly decision-relevant at the 2013 snapshot.
df['founded_at'] = pd.to_datetime(df['founded_at'], errors='coerce')
df = df[df['founded_at'].dt.year >= 2005]

## Create Target Variable

Outcomes are encoded as a binary label:

| Status | Label | Rationale |
|---|---|---|
| `acquired` | 1 | Successful exit — acquired by another company |
| `ipo` | 1 | Successful exit — went public |
| `closed` | 0 | Failed — company shut down |

`acquired` and `ipo` are merged into a single success class for simplicity. Note that not all acquisitions represent strong returns (e.g. distressed acqui-hires), so this label carries some noise — worth acknowledging as a limitation in the thesis.

In [66]:
# Create binary target variable: 1 for successful (acquired/ipo), 0 for unsuccessful (closed)
df['target'] = df['status'].apply(lambda x: 1 if x in ['acquired', 'ipo'] else 0)
df['target'].value_counts()

target
1    2045
0    1652
Name: count, dtype: int64

In [67]:
# ── Stage 4: left-join company_team_features into df (one-to-one on object_id) ─
rows_before = len(df)
df = df.merge(
    company_team_features.rename(columns={'company_id': 'object_id'}),
    on='object_id',
    how='left',
)
assert len(df) == rows_before, f"Row count changed after team join: {rows_before} → {len(df)}"
assert df['object_id'].is_unique, "Duplicate object_id after team join!"

team_cols = [
    'team_size', 'person_with_degree_count',
    'any_top_university_person', 'top_university_person_count', 'person_with_degree_pct',
]
for col in team_cols:
    df[col] = df[col].fillna(0)
df['team_size'] = df['team_size'].astype(int)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print("\nTeam feature means by outcome:")
print(df.groupby('target')[team_cols].mean().round(3).to_string())
print("\nSpot-check (top 5 by team_size):")
print(df[['name', 'target', 'team_size', 'any_top_university_person',
          'person_with_degree_pct']].sort_values('team_size', ascending=False).head(5).to_string(index=False))

Rows: 3,697 | Columns: 46

Team feature means by outcome:
        team_size  person_with_degree_count  any_top_university_person  top_university_person_count  person_with_degree_pct
target                                                                                                                     
0           2.535                     0.782                      0.150                        0.218                   0.243
1           5.448                     2.960                      0.381                        1.010                   0.409

Spot-check (top 5 by team_size):
           name  target  team_size  any_top_university_person  person_with_degree_pct
        Twitter       1        150                        1.0                0.600000
     TechCrunch       1        135                        1.0                0.733333
          Zynga       1        107                        1.0                0.710280
        Groupon       1         96                        1.0       

## Filter on `Overview` Quality

The LLM agent's primary input is the `overview` field — a free-text description of the company. Rows with very short overviews provide too little signal for meaningful text-based analysis and would make an LLM-based approach no better than a simpler ML baseline.

A minimum of **300 characters** is applied as the threshold.

In [68]:
# Filter overview column on character length >= 300
df = df[df['overview'].str.len() >= 300]

In [69]:
print(df['target'].value_counts())

target
1    1504
0    1149
Name: count, dtype: int64


## Dataset Anonymization

Columns in `objects` that could contain revealing information drop in training pipeline:
- `name`
- `normalized_name`
- `permalink`
- `homepage_url`
- `twitter_username`
- `logo_url`
- `short_description`
- `description`
- `overview`

Columns to anonymize:
- `overview`
- `short_description`

Columns to exclude from training pipeline:
- `name`
- `normalized_name`
- `permalink`
- `homepage_url`
- `twitter_username`
- `logo_url`
- `description`

### Check Information Leakage

In [70]:
# Print sample short_description and description (only print rows with non-null short_description or description)
sample = df[df['short_description'].notna() & df['description'].notna()][['object_id', 'short_description', 'description']].head(10)
print(sample.to_string(index=False))

object_id                                                                                                                            short_description                              description
  c:10176      Yammer is an enterprise social network that enables employees to collaborate across departments, geographies and business applications.                Enterprise Social Network
  c:10252                     Mob.ly is a mobile and online recommendations service providing actionable suggestions from friends and trusted sources. mobile and online recommendation service
 c:102867           WeHostels is a mobile app for young travellers to find and book accommodation, discover events, and connect with other travellers.               Youth travel mobile agency
 c:102898   Tasted Menu is a website and mobile app that allows diners to read and review restaurant dishes and make more informed ordering decisions.          Restaurant dish recommendations
 c:102922     EasyOwn.it is an e-commerc

In [71]:
# Check if description columns contain organisation names (potential data leakage)
# For simplicity, we'll just check if the company name appears in the description fields
def check_leakage(row, col='description'):
    name = str(row['name']).lower()
    desc = str(row[col]).lower() if pd.notna(row[col]) else ''
    return int(name in desc)
df['leakage_flag_description'] = df.apply(check_leakage, axis=1)
leakage_count = df['leakage_flag_description'].sum()
print(f"Rows with potential leakage (company name in description): {leakage_count} / {len(df)} ({100*leakage_count/len(df):.2f}%)")


Rows with potential leakage (company name in description): 20 / 2653 (0.75%)


In [72]:
# Check if description columns contain organisation names (potential data leakage)
# For simplicity, we'll just check if the company name appears in the description fields
df['leakage_flag_short_description'] = df.apply(lambda row: check_leakage(row=row, col='short_description'), axis=1)
leakage_count = df['leakage_flag_short_description'].sum()
print(f"Rows with potential leakage (company name in short description): {leakage_count} / {len(df)} ({100*leakage_count/len(df):.2f}%)")


Rows with potential leakage (company name in short description): 445 / 2653 (16.77%)


In [73]:
# Check if description columns contain former organisation names (potential data leakage)
def check_previous_name_leakage(row):
    string = "formerly known as"
    desc = str(row['overview']).lower() if pd.notna(row['overview']) else ''
    return int(string in desc)
df['leakage_flag_overview'] = df.apply(check_previous_name_leakage, axis=1)
leakage_count = df['leakage_flag_overview'].sum()
print(f"Rows with potential leakage (former company name in overview): {leakage_count} / {len(df)} ({100*leakage_count/len(df):.2f}%)")


Rows with potential leakage (former company name in overview): 38 / 2653 (1.43%)


In [74]:
# Remove leakage flag columns
leakage_cols = [c for c in df.columns if c.startswith('leakage_flag')]
df = df.drop(columns=leakage_cols)

### Analysis Outcome

`description` will be to cumbersome to correctly anonymize and does not provide enough value --> will be excluded from training pipeline.

`short_description` will be anonymized.

## Anonymization of `overview` and `short_description` Columns

Removes identifiable entities while preserving VC-relevant semantics.

In [75]:
# Create anonymized text using startup-name masking + contact masking.
# Keep original columns untouched and write anonymized text to `[column]_anon`.
import re

EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+\b")
PHONE_RE = re.compile(r"(?:(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)\d{3,4}[\s.-]?\d{3,4})")
DOMAIN_RE = re.compile(r"\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b")
MD_LINK_RE = re.compile(r"\[([^\]]+)\]\(([^\)]+)\)")
LEGAL_SUFFIX_RE = re.compile(
    r"(?:,?\s+(?:incorporated|inc|corp|corporation|co|company|llc|ltd|limited|plc|gmbh|sa|ag|bv))+$",
    flags=re.IGNORECASE,
 )
FORMERLY_KNOWN_AS_RE = re.compile(
    r"\s*,?\s*(?:was\s+)?formerly\s+known\s+as\s+[^,.;\n]+,?",
    flags=re.IGNORECASE,
 )
FORMERLY_RE = re.compile(r"\s*,?\s*formerly(?:\s+|,\s*)[^,.;\n]+,?", flags=re.IGNORECASE)
FORMERLY_PAREN_RE = re.compile(r"\(\s*formerly[^)]*\)", flags=re.IGNORECASE)

def _strip_markdown_links(text: str) -> str:
    # Keep visible anchor text and drop markdown URL target.
    return MD_LINK_RE.sub(r'\1', text)

# Remove phrases that indicate previous company names to prevent leakage, e.g. "formerly known as X", "X (formerly Y)", etc.
def _remove_previous_name_phrases(text: str) -> str:
    # Remove patterns like "formerly ...", "formerly known as ...", and common variants.
    text = FORMERLY_KNOWN_AS_RE.sub(' ', text)
    text = FORMERLY_PAREN_RE.sub(' ', text)
    text = FORMERLY_RE.sub(' ', text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    return text.strip()

# Build robust startup-name variants for matching in noisy text, e.g. with/without legal suffixes, punctuation replaced by whitespace, etc.
def _org_name_variants(org_name: str):
    """Build robust startup-name variants for matching in noisy text."""
    variants = set()
    base = org_name.strip()
    if not base:
        return variants

    variants.add(base)

    # Remove trailing legal suffixes: "Inc.", "LLC", etc.
    no_suffix = LEGAL_SUFFIX_RE.sub('', base).strip(' ,.-')
    if no_suffix:
        variants.add(no_suffix)

    # If name is a bare domain, also add leftmost label as potential brand token.
    if re.fullmatch(r"(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}", base):
        label = base.split('.')[0].strip()
        if label:
            variants.add(label)

    # If the name includes punctuation, also try a whitespace-normalized variant.
    normalized = re.sub(r"[^A-Za-z0-9]+", " ", no_suffix if no_suffix else base).strip()
    if normalized:
        variants.add(normalized)

    # Keep meaningful variants only.
    return {v for v in variants if len(v) >= 3}

# Mask startup name variants in text with [ORG] placeholder
def _mask_startup_name(text: str, org_name) -> str:
    if not isinstance(org_name, str):
        return text

    variants = sorted(_org_name_variants(org_name), key=len, reverse=True)
    if not variants:
        return text

    masked = text
    for variant in variants:
        # Token-safe boundaries reduce accidental partial-word replacements.
        pattern = re.compile(
            rf"(?<![A-Za-z0-9]){re.escape(variant)}(?![A-Za-z0-9])",
            flags=re.IGNORECASE,
        )
        masked = pattern.sub('[ORG]', masked)
    return masked

def _mask_contacts(text: str) -> str:
    # Keep startup-name masking priority by running this after _mask_startup_name.
    masked = EMAIL_RE.sub('[EMAIL]', text)
    masked = URL_RE.sub('[URL]', masked)
    masked = PHONE_RE.sub('[PHONE]', masked)
    masked = DOMAIN_RE.sub('[URL]', masked)
    return masked

def anonymize_text(text, org_name=None):
    if not isinstance(text, str):
        return text
    text = text.strip()
    if not text:
        return text

    text = _strip_markdown_links(text)
    text = _remove_previous_name_phrases(text)
    text = _mask_startup_name(text, org_name)
    text = _mask_contacts(text)
    return text

name_col = 'object_name' if 'object_name' in df.columns else ('name' if 'name' in df.columns else None)
text_cols = ['overview', 'short_description']

if name_col is None:
    print("Warning: neither 'object_name' nor 'name' exists; masking cannot target startup names.")
    for col in text_cols:
        if col in df.columns:
            df[f'{col}_anon'] = [anonymize_text(text) for text in df[col]]
else:
    for col in text_cols:
        if col in df.columns:
            df[f'{col}_anon'] = [
                anonymize_text(text, org_name=org_name)
                for text, org_name in zip(df[col], df[name_col])
            ]

print('Anonymization complete.')
print(f"Rows: {len(df)}")
print(f"Org-name source column: {name_col}")
for col in text_cols:
    if f'{col}_anon' in df.columns:
        print(f"Created column: {col}_anon")

sample_cols = [c for c in ['overview', 'overview_anon', 'short_description', 'short_description_anon'] if c in df.columns]
sample = df[sample_cols].dropna().head(3)
for i, row in sample.iterrows():
    print(f"\n--- sample index {i} ---")
    if 'overview' in sample_cols:
        print('ORIGINAL OVERVIEW:')
        print(row['overview'][:400])
    if 'overview_anon' in sample_cols:
        print('\nANONYMIZED OVERVIEW:')
        print(row['overview_anon'][:400])
    if 'short_description' in sample_cols:
        print('\n--- short description ---')
        print(row['short_description'][:400])
    if 'short_description_anon' in sample_cols:
        print('\n--- short description anonymized ---')
        print(row['short_description_anon'][:400])

Anonymization complete.
Rows: 2653
Org-name source column: name
Created column: overview_anon
Created column: short_description_anon

--- sample index 19 ---
ORIGINAL OVERVIEW:
Yammer (www.yammer.com) is an Enterprise Social Network that brings together employees, content, conversations, and business data in a single location. Built for the entrprise and loved by users, Yammer empowers employees to be more productive by enabling them to collaborate in real-time across departments, geographies, and business applications. Employees can create groups to collaborate on proje

ANONYMIZED OVERVIEW:
[ORG] ([URL]) is an Enterprise Social Network that brings together employees, content, conversations, and business data in a single location. Built for the entrprise and loved by users, [ORG] empowers employees to be more productive by enabling them to collaborate in real-time across departments, geographies, and business applications. Employees can create groups to collaborate on projects and sha

## Data Exploration

## Missing Value Analysis

In [76]:
# Print full missing-values summary 
missing_summary = df.isnull().sum().sort_values(ascending=False)
print(missing_summary.to_string())

parent_id                      2653
last_investment_at             2630
first_investment_at            2630
short_description              2148
short_description_anon         2148
closed_at                      1489
state_code                     1014
created_by                      946
twitter_username                926
first_funding_at                849
last_funding_at                 849
first_milestone_at              804
last_milestone_at               804
tag_list                        731
description                     629
city                            403
country_code                    335
logo_url                        105
category_code                    52
domain                           34
homepage_url                     34
logo_width                        0
normalized_name                   0
permalink                         0
entity_type                       0
status                            0
founded_at                        0
object_id                   

### Missing Value Handling and Feature Policy

**Deferred to training pipeline (fit on train split only):**
- `category_code`, `country_code`, `city`: fill missing with `"unknown"`
- `founded_at`: median imputation + explicit missing indicator
- `first_funding_at`, `last_funding_at`: create funding-date-presence indicator(s)

**Dropped in processing phase:**
- Very high missingness / weak signal: `parent_id`, `tag_list`, `twitter_username`
- Outcome leakage: `closed_at`, `last_investment_at`, `first_investment_at`
- Redundant / low-value metadata and identifiers not needed by agents

This notebook performs only deterministic cleaning and column selection. Statistical imputations are intentionally deferred.

### Apply Column Selection

Drop the columns identified above and cast `founded_at` to datetime. No imputation is applied — the planned strategies are documented in comments and will be implemented inside a `sklearn` pipeline fitted only on the training split.

In [77]:
# Type casting (safe: no statistics derived from data)
df['founded_at'] = pd.to_datetime(df['founded_at'], errors='coerce')

# Drop columns: high missingness, leakage, or pure metadata.
# Imputation decisions for remaining columns are deferred to training-time pipelines.
cols_to_drop = [
    'parent_id',            # 99% missing
    'last_investment_at',   # 96% missing and leakage-prone
    'first_investment_at',  # 96% missing and leakage-prone
    'closed_at',            # leakage: directly encodes outcome timing
    'tag_list',             # 62% missing
    'twitter_username',     # 60% missing
    'first_milestone_at',   # sparse, milestones count retained
    'last_milestone_at',    # sparse, milestones count retained
    'state_code',           # redundant with broader location features
    'logo_url', 'logo_width', 'logo_height',
    'domain', 'homepage_url', 'permalink',
    'normalized_name', 'entity_id', 'created_by', 'description',  
]
drop_candidates = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=drop_candidates)

print(f"Rows: {len(df)} | Columns: {df.shape[1]}")
print(f"Dropped columns ({len(drop_candidates)}):")
for col in drop_candidates:
    print(f"  - {col}")

print("\nRemaining missing values (to be handled in training pipeline):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].to_string())

Rows: 2653 | Columns: 29
Dropped columns (19):
  - parent_id
  - last_investment_at
  - first_investment_at
  - closed_at
  - tag_list
  - twitter_username
  - first_milestone_at
  - last_milestone_at
  - state_code
  - logo_url
  - logo_width
  - logo_height
  - domain
  - homepage_url
  - permalink
  - normalized_name
  - entity_id
  - created_by
  - description

Remaining missing values (to be handled in training pipeline):
short_description         2148
short_description_anon    2148
first_funding_at           849
last_funding_at            849
city                       403
country_code               335
category_code               52


## Save Final Processed Dataset

In [78]:
output_path = "../data/processed/companies_clean.parquet"
df.to_parquet(output_path, index=False)

print(f"Saved {len(df)} rows, {df.shape[1]} columns")
print(f"Path: {output_path}")

required_cols = ['object_id', 'status', 'target', 'overview', 'overview_anon']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns in final dataset: {missing_required}")

print("\nFinal class balance:")
print(df['target'].value_counts(dropna=False).to_string())
print(f"Success rate: {df['target'].mean():.2%}")

print("\nCritical-column missingness:")
critical_cols = ['overview', 'overview_anon', 'category_code', 'country_code', 'city', 'founded_at']
for col in critical_cols:
    if col in df.columns:
        miss = int(df[col].isna().sum())
        miss_pct = 100 * miss / len(df) if len(df) else 0
        print(f"  {col:14s} {miss:6d} ({miss_pct:5.2f}%)")

print("\nSchema preview:")
for col in df.columns:
    print(f"  {col:25s} {str(df[col].dtype)}")

Saved 2653 rows, 29 columns
Path: ../data/processed/companies_clean.parquet

Final class balance:
target
1    1504
0    1149
Success rate: 56.69%

Critical-column missingness:
  overview            0 ( 0.00%)
  overview_anon       0 ( 0.00%)
  category_code      52 ( 1.96%)
  country_code      335 (12.63%)
  city              403 (15.19%)
  founded_at          0 ( 0.00%)

Schema preview:
  object_id                 object
  entity_type               object
  name                      object
  category_code             object
  status                    object
  founded_at                datetime64[us]
  short_description         object
  overview                  object
  country_code              object
  city                      object
  region                    object
  investment_rounds         int64
  invested_companies        int64
  first_funding_at          object
  last_funding_at           object
  funding_rounds            int64
  funding_total_usd         float64
  milest